# 11 — Additional Paper Figures

Generates four additional result figures for the paper:

1. **LR and KNN classification accuracy** — time-windowed decoding across layers
2. **Activation distributions / maxima** — microns natural video vs flow stimuli across layers
3. **Representational alignment metrics** — RSA, CCA, linear alignment, DSA across layers
4. **OSI and DSI** — FNN orientation/direction selectivity, pink noise vs flow stimuli

Code adapted from `additional/` scripts with path fixes only.

In [ ]:
import sys, os
sys.path.insert(0, '..')
os.makedirs('../fig/additional', exist_ok=True)


## 1. LR and KNN Classification Accuracy

Time-windowed stimulus classification (increasing and decreasing windows) for Encoder L1, Encoder L8, and Recurrent layers using Logistic Regression and k-NN (k=3).

Source: `additional/neurips_plots.py` → `create_knn_lr_plots()`  
Output: `fig/additional/knn_lr_combined.pdf`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
import os

PREFIXES = [
    'fnn07_act_i3_n2000_SCL0_7_TL37_inputs0_maxFr_maxNr_seed1',
    'fnn07_act_i3_n2000_SCL0_7_TL37_blocks0_maxFr_maxNr_seed1',
    'fnn07_act_i3_n2000_SCL0_7_TL37_inputs1_maxFr_maxNr_seed1',
    'fnn07_act_i3_n2000_SCL0_7_TL37_blocks1_maxFr_maxNr_seed1',
    'fnn07_act_i3_n2000_SCL0_7_TL37_inputs2_maxFr_maxNr_seed1',
    'fnn07_act_i3_n2000_SCL0_7_TL37_blocks2_maxFr_maxNr_seed1',
    'fnn07_act_i3_n2000_SCL0_7_TL37_hidden_maxFr_maxNr_seed1',
    'fnn07_act_i3_n2000_SCL0_7_TL37_recurrentout_maxFr_maxNr_seed1',
    'fnn07_act_i3_n2000_SCL0_7_TL37_position_maxFr_maxNr_seed1',
    'fnn07_seed2',
]
LAYER_NAMES = [
    'Encoder L1', 'Encoder L2', 'Encoder L4', 'Encoder L5', 'Encoder L7',
    'Encoder L8', 'Recurrent', 'Recurrent-Out', 'Readout', 'Output',
]

basedir   = '../data/sampled'
cache_dir = '../data/lr_cache'
os.makedirs(cache_dir, exist_ok=True)

n_runs      = 5
k_neighbors = 3
loocv       = LeaveOneOut()

lr_inc, lr_dec, lr_inc_std, lr_dec_std = [], [], [], []
knn_inc, knn_dec = [], []
T_len = None

for idx, prefix in enumerate(PREFIXES):
    tensor4d = np.load(f'{basedir}/tensor4d_{prefix}.npy')  # (N, S, D, T)
    N, n_stims, n_dirs, T_len = tensor4d.shape
    n_samples = n_stims * n_dirs
    # NOTE: after np.transpose(tensor4d, (2,1,0,3)) the layout is (D,S,N,T),
    # so when reshaped to (n_samples, N) the sample order is direction-major.
    # Correct labels use tile (not repeat): [0,1,...,10, 0,1,...,10, ...]
    y = np.array(list(range(n_stims)) * n_dirs)

    # ---- Logistic Regression (cached per prefix) ----
    cf_im = os.path.join(cache_dir, f'{prefix}_inc_mean.npy')
    cf_dm = os.path.join(cache_dir, f'{prefix}_dec_mean.npy')
    cf_is = os.path.join(cache_dir, f'{prefix}_inc_std.npy')
    cf_ds = os.path.join(cache_dir, f'{prefix}_dec_std.npy')

    if all(os.path.exists(f) for f in [cf_im, cf_dm, cf_is, cf_ds]):
        print(f'{LAYER_NAMES[idx]}: loading cached LR')
        inc_m, dec_m = np.load(cf_im), np.load(cf_dm)
        inc_s, dec_s = np.load(cf_is), np.load(cf_ds)
    else:
        print(f'{LAYER_NAMES[idx]}: computing LR ...')
        runs_inc, runs_dec = [], []
        for run in range(n_runs):
            clf = LogisticRegression(max_iter=500, solver='lbfgs', random_state=42 + run)
            cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42 + run)
            t_inc, t_dec = [], []
            for i in range(T_len):
                Xi = np.transpose(tensor4d, (2, 1, 0, 3))[:, :, :, :i+1].mean(axis=3).reshape(n_samples, -1)
                t_inc.append(cross_val_score(clf, Xi, y, cv=cv).mean())
                Xd = np.transpose(tensor4d, (2, 1, 0, 3))[:, :, :, i:].mean(axis=3).reshape(n_samples, -1)
                t_dec.append(cross_val_score(clf, Xd, y, cv=cv).mean())
            runs_inc.append(t_inc)
            runs_dec.append(t_dec)
        inc_m = np.mean(runs_inc, axis=0); inc_s = np.std(runs_inc, axis=0)
        dec_m = np.mean(runs_dec, axis=0); dec_s = np.std(runs_dec, axis=0)
        np.save(cf_im, inc_m); np.save(cf_dm, dec_m)
        np.save(cf_is, inc_s); np.save(cf_ds, dec_s)

    lr_inc.append(inc_m); lr_dec.append(dec_m)
    lr_inc_std.append(inc_s); lr_dec_std.append(dec_s)

    # ---- KNN (cached) ----
    cf_ki = os.path.join(cache_dir, f'{prefix}_knn_inc.npy')
    cf_kd = os.path.join(cache_dir, f'{prefix}_knn_dec.npy')
    if os.path.exists(cf_ki) and os.path.exists(cf_kd):
        print(f'{LAYER_NAMES[idx]}: loading cached KNN')
        t_inc_arr, t_dec_arr = np.load(cf_ki), np.load(cf_kd)
    else:
        print(f'{LAYER_NAMES[idx]}: computing KNN ...')
        t_inc_arr, t_dec_arr = [], []
        for i in range(T_len):
            Xi = np.transpose(tensor4d, (2, 1, 0, 3))[:, :, :, :i+1].mean(3).reshape(n_samples, -1)
            Xd = np.transpose(tensor4d, (2, 1, 0, 3))[:, :, :, i:].mean(3).reshape(n_samples, -1)
            for X_w, storage in [(Xi, t_inc_arr), (Xd, t_dec_arr)]:
                preds, trues = [], []
                for tr, te in loocv.split(X_w):
                    knn = KNeighborsClassifier(n_neighbors=k_neighbors)
                    knn.fit(X_w[tr], y[tr])
                    preds.append(knn.predict(X_w[te])[0])
                    trues.append(y[te][0])
                storage.append(accuracy_score(trues, preds))
        np.save(cf_ki, t_inc_arr); np.save(cf_kd, t_dec_arr)

    knn_inc.append(t_inc_arr); knn_dec.append(t_dec_arr)

# ---- Plot: 2×5 per-layer grid matching paper Figure 8 ----
from tueplots import bundles
from tueplots import axes as tpaxes

time_pts = np.arange(T_len)  # 0-indexed: 0..36

with plt.rc_context({**bundles.neurips2024(), **tpaxes.lines()}):
    fig = plt.figure(figsize=(15, 12))
    sf_lr, sf_knn = fig.subfigures(2, 1, hspace=0.08)

    # ---- LR subfigure ----
    sf_lr.suptitle('Logistic Regression (LR)', fontsize=14, fontweight='bold', y=1.01)
    axs_lr = sf_lr.subplots(2, 5, gridspec_kw=dict(hspace=0.6, wspace=0.35))

    for j, (ax, name) in enumerate(zip(axs_lr.flat, LAYER_NAMES)):
        m_inc = np.asarray(lr_inc[j])
        s_inc = np.asarray(lr_inc_std[j])
        m_dec = np.asarray(lr_dec[j])
        s_dec = np.asarray(lr_dec_std[j])
        sem_inc = s_inc / np.sqrt(n_runs)
        sem_dec = s_dec / np.sqrt(n_runs)

        ax.plot(time_pts, m_inc, color='red',  lw=1.2, marker='o', markersize=2,
                label='Increasing (Start→t)', alpha=0.9)
        ax.fill_between(time_pts, m_inc - sem_inc, m_inc + sem_inc, color='red',  alpha=0.2)

        ax.plot(time_pts, m_dec, color='blue', lw=1.2, marker='s', markersize=2,
                label='Decreasing (t→End)', alpha=0.9)
        ax.fill_between(time_pts, m_dec - sem_dec, m_dec + sem_dec, color='blue', alpha=0.2)

        ax.set_ylim(0.0, 1.0)
        ax.set_title(name, fontsize=8, fontweight='bold')
        row, col = divmod(j, 5)
        if row == 1:
            ax.set_xlabel('Time Point', fontsize=7)
        if col == 0:
            ax.set_ylabel('Accuracy', fontsize=7)
        if j == 0:
            ax.legend(loc='lower right', fontsize=5)

    # ---- KNN subfigure ----
    sf_knn.suptitle('K-Nearest Neighbors (KNN)', fontsize=14, fontweight='bold', y=1.01)
    axs_knn = sf_knn.subplots(2, 5, gridspec_kw=dict(hspace=0.6, wspace=0.35))

    for j, (ax, name) in enumerate(zip(axs_knn.flat, LAYER_NAMES)):
        ax.plot(time_pts, np.asarray(knn_inc[j]), color='red',  lw=1.2, marker='o', markersize=2,
                label='Increasing (Start→t)', alpha=0.9)
        ax.plot(time_pts, np.asarray(knn_dec[j]), color='blue', lw=1.2, marker='s', markersize=2,
                label='Decreasing (t→End)', alpha=0.9)

        ax.set_ylim(0.0, 1.0)
        ax.set_title(name, fontsize=8, fontweight='bold')
        row, col = divmod(j, 5)
        if row == 1:
            ax.set_xlabel('Time Point', fontsize=7)
        if col == 0:
            ax.set_ylabel('Accuracy', fontsize=7)
        if j == 0:
            ax.legend(loc='lower right', fontsize=5)

    os.makedirs('../fig/additional', exist_ok=True)
    plt.savefig('../fig/additional/knn_lr_combined.pdf', bbox_inches='tight')
    plt.savefig('../fig/additional/knn_lr_combined.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved → ../fig/additional/knn_lr_combined.pdf')


## 2. Activation Function Output Distributions / Maxima

Compares maximum activations across FNN layers for natural video (microns MP4) vs optical-flow stimuli.

**Requires:** `fnn` package, `../data/stimulus_17797_7_3_v4_compressed.mp4`  
Source: `additional/neurips_plots.py` → `intensity_comparison_plot()`  
Output: `fig/additional/intensity_comparison_microns.png`

In [ ]:
import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from fnn import microns
from src.plot_utils import createFlowDataset

LAYER_HOOKS = [
    'core.feedforward.inputs.0',
    'core.feedforward.inputs.1',
    'core.feedforward.inputs.2',
    'core.feedforward.blocks.0.convs.1',
    'core.feedforward.blocks.1.convs.1',
    'core.feedforward.blocks.2.convs.2',
    'core.recurrent.conv',
    'core.recurrent.out',
    'readout',
    'unit',
]
LAYER_LABELS = [
    'Encoder L1', 'Encoder L2', 'Encoder L4',
    'Encoder L5', 'Encoder L7', 'Encoder L8',
    'Recurrent', 'Recurrent-Out', 'Readout', 'Output',
]


def get_activation_distributions(frames, model_session=8, model_scan=5):
    model, ids = microns.scan(session=model_session, scan_idx=model_scan)
    history = {name: [] for name in LAYER_HOOKS}

    def make_hook(name):
        def hook(module, inp, out):
            act = out.detach().cpu().float().numpy()
            if act.ndim == 4:           # (B, C, H, W)
                act = act.max(axis=(-2, -1))  # -> (B, C)
            elif act.ndim == 3:         # (B, C, L)
                act = act.max(axis=-1)        # -> (B, C)
            elif act.ndim == 1:
                act = act[np.newaxis, :]
            history[name].append(act.reshape(-1, act.shape[-1]))
        return hook

    handles = []
    for name, module in model.named_modules():
        if name in LAYER_HOOKS:
            handles.append(module.register_forward_hook(make_hook(name)))

    with torch.no_grad():
        model.predict(stimuli=frames)

    for h in handles:
        h.remove()

    per_unit_max = {}
    for name in LAYER_HOOKS:
        if history[name]:
            stacked = np.vstack(history[name])        # (T, units)
            per_unit_max[name] = stacked.max(axis=0)  # (units,)
    return per_unit_max


# ---- Load stimuli ----
cap = cv2.VideoCapture('../data/stimulus_17797_7_3_v4_compressed.mp4')
mp4_frames = []
counter = 0
while counter < 10000:
    counter += 1
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    mp4_frames.append(cv2.resize(gray, (256, 144)))
cap.release()
mp4_frames = np.array(mp4_frames, dtype='uint8')

categories = [
    'grat_W12', 'grat_W1', 'grat_W2',
    'neg1dotflow_D1_bg', 'neg3dotflow_D1_bg', 'neg1dotflow_D2_bg', 'neg3dotflow_D2_bg',
    'pos1dotflow_D1_bg', 'pos3dotflow_D1_bg', 'pos1dotflow_D2_bg', 'pos3dotflow_D2_bg',
]
mydirs = list(map(str, range(0, 360, 45)))
flow_ds = createFlowDataset(categories, '../stimuli/flowstims', mydirs,
                            (800, 600), (144, 256), 0.7, 3, 37, 1)
flow_frames = np.array([f.reshape(144, 256) for f in flow_ds[0][:1000]], dtype='uint8')

# ---- Compute per-unit max distributions ----
print('Computing MICrONS activations...')
mp4_dist  = get_activation_distributions(mp4_frames)
print('Computing flow activations...')
flow_dist = get_activation_distributions(flow_frames)

# ---- Build arrays ----
n_layers = len(LAYER_HOOKS)
x = np.arange(n_layers)
offset = 0.18

mp4_maxes,  flow_maxes  = [], []
mp4_data,   flow_data   = [], []
for name in LAYER_HOOKS:
    mv = mp4_dist.get(name,  np.array([0.0]))
    fv = flow_dist.get(name, np.array([0.0]))
    mp4_data.append(mv);  flow_data.append(fv)
    mp4_maxes.append(mv.max()); flow_maxes.append(fv.max())

# ---- Plot ----
fig, ax = plt.subplots(figsize=(16, 6))

bp_kw = dict(
    widths=0.28, patch_artist=True, showfliers=True,
    flierprops=dict(marker='o', markersize=2, linestyle='none', markeredgecolor='k'),
    medianprops=dict(color='k', linewidth=1.5),
    whiskerprops=dict(color='k'),
    capprops=dict(color='k'),
    boxprops=dict(linewidth=0.8),
)
for i, (mv, fv) in enumerate(zip(mp4_data, flow_data)):
    bp1 = ax.boxplot(mv, positions=[x[i] - offset], **bp_kw)
    bp2 = ax.boxplot(fv, positions=[x[i] + offset], **bp_kw)
    bp1['boxes'][0].set_facecolor((1.0, 0.7, 0.7, 0.5))
    bp2['boxes'][0].set_facecolor((0.7, 0.7, 1.0, 0.5))

ax.plot(x, mp4_maxes,  'o-', color='red',  linewidth=2, markersize=5, zorder=5,
        label='MICrONS Natural Stimuli (Max)')
ax.plot(x, flow_maxes, 's-', color='blue', linewidth=2, markersize=5, zorder=5,
        label='Flow Stimuli (Max)')

ax.legend(handles=[
    plt.Line2D([0],[0], color='red',  marker='o', linewidth=2, label='MICrONS Natural Stimuli (Max)'),
    plt.Line2D([0],[0], color='blue', marker='s', linewidth=2, label='Flow Stimuli (Max)'),
    Patch(facecolor=(1.0, 0.7, 0.7, 0.6), edgecolor='k', label='MICrONS Distribution'),
    Patch(facecolor=(0.7, 0.7, 1.0, 0.6), edgecolor='k', label='Flow Distribution'),
], fontsize=10, loc='upper left')

ax.set_xticks(x)
ax.set_xticklabels(LAYER_LABELS, rotation=15, ha='right', fontsize=10)
ax.set_ylabel('Activation Values', fontsize=12)
ax.set_title('Activation Distribution per Layer: MICrONS vs Flow Stimuli', fontsize=12)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../fig/additional/intensity_comparison_microns.png', dpi=200, bbox_inches='tight')
plt.savefig('../fig/additional/intensity_comparison_microns.pdf', bbox_inches='tight')
plt.show()
print('Saved → ../fig/additional/intensity_comparison_microns.png')


## 3. Representational Alignment Metrics

Computes RSA, CCA, linear alignment (LP), and DSA between FNN layers and biological data (V1, Retina) across 9 FNN layers. Produces time-resolved plots and a summary layer-by-layer comparison.

Source: `additional/metrics.py`  
Output: `fig/additional/neural_metrics_comparison_v1.png`, `fig/additional/neural_metrics_comparison_retina.png`, `fig/additional/neural_metrics_by_layer.png`

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning, module='sklearn')

import numpy as np
import matplotlib.pyplot as plt
import time
from scipy.signal import resample
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import CCA
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

try:
    from DSA import DSA
    HAS_DSA = True
except ImportError:
    HAS_DSA = False
    print('DSA package not found — DSA scores will be NaN. Install with: pip install dsa-analysis')

# ── Config ──────────────────────────────────────────────────────────────────
# True  → time-average neural responses first, then compute metrics once (higher SNR)
# False → compute metrics at every time step, then average scores across time
#         (matches additional/metrics.py; useful for comparing the two approaches)
TIME_AVERAGE = False
# ────────────────────────────────────────────────────────────────────────────


# --- Utility ---

def resample_and_reshape(data, target_time_steps):
    n_neurons, n_stim, n_dir, n_time = data.shape
    n_conditions = n_stim * n_dir
    data_reshaped = data.reshape(n_neurons, n_conditions, n_time)
    if n_time != target_time_steps:
        data_resampled = resample(data_reshaped, target_time_steps, axis=2, window='hamming')
    else:
        data_resampled = data_reshaped
    return data_resampled.transpose(1, 0, 2)   # (n_cond, n_neurons, T)


def compute_rsa_at_t(data1_t, data2_t):
    rdm1 = 1.0 - np.corrcoef(data1_t)
    rdm2 = 1.0 - np.corrcoef(data2_t)
    upper = np.triu_indices_from(rdm1, k=1)
    return spearmanr(rdm1[upper], rdm2[upper])[0]


def compute_cca_at_t(data1_t, data2_t, n_components=3):
    n_cond = data1_t.shape[0]
    n_comp_safe = min(n_components, n_cond // 8, data1_t.shape[1], data2_t.shape[1])
    if n_comp_safe < 3:
        return np.nan, np.nan, np.nan
    pca1 = PCA(n_components=n_comp_safe)
    pca2 = PCA(n_components=n_comp_safe)
    data1_pca = pca1.fit_transform(data1_t)
    data2_pca = pca2.fit_transform(data2_t)
    n_cca_comp = min(3, n_comp_safe)
    cca = CCA(n_components=n_cca_comp)
    cca.fit(data1_pca, data2_pca)
    X_c, Y_c = cca.transform(data1_pca, data2_pca)
    corrs = [np.corrcoef(X_c[:, i], Y_c[:, i])[0, 1] for i in range(n_cca_comp)]
    return np.mean(corrs), pca1.explained_variance_ratio_.sum(), pca2.explained_variance_ratio_.sum()


def compute_manifold_alignment_at_t(model_data_t, brain_data_t, n_model_pca=10, n_brain_pca=2):
    n_conditions = brain_data_t.shape[0]
    model_data_scaled = StandardScaler().fit_transform(model_data_t)
    brain_data_scaled = StandardScaler().fit_transform(brain_data_t)
    n_model_pca_safe = min(n_model_pca, n_conditions - 1, model_data_t.shape[1])
    n_brain_pca_safe = min(n_brain_pca, n_conditions - 1, brain_data_t.shape[1])
    if n_model_pca_safe < 1 or n_brain_pca_safe < 1:
        return np.nan
    model_pca_data = PCA(n_components=n_model_pca_safe).fit_transform(model_data_scaled)
    brain_pca_data = PCA(n_components=n_brain_pca_safe).fit_transform(brain_data_scaled)
    scores = []
    for j in range(10):
        shuffle = np.arange(n_conditions)
        np.random.shuffle(shuffle)
        test = shuffle[:max(1, n_conditions // 6)]
        train = shuffle[max(1, n_conditions // 6):]
        for i in range(brain_pca_data.shape[1]):
            m = Ridge(alpha=1)
            m.fit(model_pca_data[train], brain_pca_data[train, i])
            scores.append(np.clip(m.score(model_pca_data[test], brain_pca_data[test, i]), 0, 1))
    return np.mean(scores)


def compute_metrics_tavg(data1, data2):
    """Metrics on time-averaged responses (n_cond, n_neurons). Higher SNR, single evaluation."""
    d1 = data1.mean(axis=2)
    d2 = data2.mean(axis=2)
    rsa = compute_rsa_at_t(d1, d2)
    cca_val = compute_cca_at_t(d1, d2)
    cca = cca_val[0] if not np.isnan(cca_val[0]) else 0.0
    enc = compute_manifold_alignment_at_t(d2, d1)
    return rsa, cca, enc


def compute_metrics_tresolved(data1, data2):
    """Metrics computed at every time step, scores averaged across time.
    Matches additional/metrics.py aggregation strategy."""
    T = data1.shape[2]
    rsa_scores, cca_scores, enc_scores = [], [], []
    for t in range(T):
        d1t = data1[:, :, t]
        d2t = data2[:, :, t]
        rsa_scores.append(compute_rsa_at_t(d1t, d2t))
        cca_val = compute_cca_at_t(d1t, d2t)
        cca_scores.append(cca_val[0] if not np.isnan(cca_val[0]) else 0.0)
        enc_scores.append(compute_manifold_alignment_at_t(d2t, d1t))
    return float(np.mean(rsa_scores)), float(np.mean(cca_scores)), float(np.mean(enc_scores))


def compute_metrics(data1, data2):
    """Dispatcher: uses TIME_AVERAGE flag to pick aggregation strategy."""
    if TIME_AVERAGE:
        return compute_metrics_tavg(data1, data2)
    else:
        return compute_metrics_tresolved(data1, data2)


def compute_dsa(data1, data2, n_components=10, n_permutations=50):
    """Returns (z_score, 1 - true_dsa_score). z_score > 0 means more similar than shuffled."""
    if not HAS_DSA:
        return np.nan, np.nan
    N_trials = data1.shape[0]
    data1_t = np.transpose(data1, (0, 2, 1))
    data2_t = np.transpose(data2, (0, 2, 1))
    data1_flat = data1_t.reshape(-1, data1_t.shape[-1])
    data2_flat = data2_t.reshape(-1, data2_t.shape[-1])
    pca1 = PCA(n_components=n_components)
    data1_pca_flat = pca1.fit_transform(data1_flat)
    data1_reduced = data1_pca_flat.reshape(N_trials, data1_t.shape[1], n_components)
    pca2 = PCA(n_components=n_components)
    data2_pca_flat = pca2.fit_transform(data2_flat)
    data2_reduced_true = data2_pca_flat.reshape(N_trials, data2_t.shape[1], n_components)
    dsa_true = DSA(X=data1_reduced, Y=data2_reduced_true, n_delays=1)
    true_dsa_score = np.mean(dsa_true.fit_score())
    null_scores = []
    for _ in range(n_permutations):
        shuffled = np.random.permutation(data2_t.shape[1])
        dsa_perm = DSA(X=data1_reduced, Y=data2_reduced_true[:, shuffled, :], n_delays=1)
        null_scores.append(np.mean(dsa_perm.fit_score()))
    null_scores = np.array(null_scores)
    z_score = (true_dsa_score - np.mean(null_scores)) / np.std(null_scores)
    return -z_score, 1 - true_dsa_score


# --- Load data ---
T_TARGET = 37
basedir  = '../data/sampled'

retina = np.load(f'{basedir}/tensor4d_retina.npy')[:, :6]
v1     = np.load(f'{basedir}/tensor4d_V1.npy')[:, :6]
model1 = np.load(f'{basedir}/tensor4d_fnn07_act_i3_n2000_SCL0_7_TL37_inputs0_maxFr_maxNr_seed1.npy')[:, :6]
model2 = np.load(f'{basedir}/tensor4d_fnn07_act_i3_n2000_SCL0_7_TL37_blocks0_maxFr_maxNr_seed1.npy')[:, :6]
model3 = np.load(f'{basedir}/tensor4d_fnn07_act_i3_n2000_SCL0_7_TL37_inputs1_maxFr_maxNr_seed1.npy')[:, :6]
model4 = np.load(f'{basedir}/tensor4d_fnn07_act_i3_n2000_SCL0_7_TL37_blocks1_maxFr_maxNr_seed1.npy')[:, :6]
model5 = np.load(f'{basedir}/tensor4d_fnn07_act_i3_n2000_SCL0_7_TL37_inputs2_maxFr_maxNr_seed1.npy')[:, :6]
model6 = np.load(f'{basedir}/tensor4d_fnn07_act_i3_n2000_SCL0_7_TL37_blocks2_maxFr_maxNr_seed1.npy')[:, :6]
model7 = np.load(f'{basedir}/tensor4d_fnn07_act_i3_n2000_SCL0_7_TL37_hidden_maxFr_maxNr_seed1.npy')[:, :6]
model8 = np.load(f'{basedir}/tensor4d_fnn07_act_i3_n2000_SCL0_7_TL37_position_maxFr_maxNr_seed1.npy')[:, :6]
model9 = np.load(f'{basedir}/tensor4d_fnn07_seed2.npy')[:, :6]

retina_prep = resample_and_reshape(retina, T_TARGET)
v1_prep     = resample_and_reshape(v1,     T_TARGET)
model1_prep = resample_and_reshape(model1, T_TARGET)
model2_prep = resample_and_reshape(model2, T_TARGET)
model3_prep = resample_and_reshape(model3, T_TARGET)
model4_prep = resample_and_reshape(model4, T_TARGET)
model5_prep = resample_and_reshape(model5, T_TARGET)
model6_prep = resample_and_reshape(model6, T_TARGET)
model7_prep = resample_and_reshape(model7, T_TARGET)
model8_prep = resample_and_reshape(model8, T_TARGET)
model9_prep = resample_and_reshape(model9, T_TARGET)
print(f'Data loaded and reshaped.  TIME_AVERAGE={TIME_AVERAGE}')

comparisons = {
    'V1 vs L1':          (v1_prep,     model1_prep),
    'V1 vs L3':          (v1_prep,     model2_prep),
    'V1 vs L6':          (v1_prep,     model3_prep),
    'V1 vs L8':          (v1_prep,     model4_prep),
    'V1 vs L11':         (v1_prep,     model5_prep),
    'V1 vs L13':         (v1_prep,     model6_prep),
    'V1 vs Rec':         (v1_prep,     model7_prep),
    'V1 vs Readout':     (v1_prep,     model8_prep),
    'V1 vs Output':      (v1_prep,     model9_prep),
    'Retina vs L1':      (retina_prep, model1_prep),
    'Retina vs L3':      (retina_prep, model2_prep),
    'Retina vs L6':      (retina_prep, model3_prep),
    'Retina vs L8':      (retina_prep, model4_prep),
    'Retina vs L11':     (retina_prep, model5_prep),
    'Retina vs L13':     (retina_prep, model6_prep),
    'Retina vs Rec':     (retina_prep, model7_prep),
    'Retina vs Readout': (retina_prep, model8_prep),
    'Retina vs Output':  (retina_prep, model9_prep),
}

# --- Time-resolved metrics (disabled — set to True to run) ---
results = {}
time_points = np.arange(T_TARGET)

if False:  # set True to compute time-resolved RSA/CCA/encoding/DSA per layer
    print('Running time-resolved metrics...')
    for name, (data1, data2) in comparisons.items():
        print(f'  {name}')
        dsa_z, dsa_score = compute_dsa(data1, data2)
        rsa_scores, cca_scores, encoding_scores = [], [], []
        for t in time_points:
            data1_t = data1[:, :, t]
            data2_t = data2[:, :, t]
            if t < len(time_points) - 4:
                d1m = data1[:, :, t:t+3].mean(axis=2)
                d2m = data2[:, :, t:t+3].mean(axis=2)
            else:
                d1m = data1[:, :, t-3:t].mean(axis=2)
                d2m = data2[:, :, t-3:t].mean(axis=2)
            rsa_scores.append(compute_rsa_at_t(data1_t, data2_t))
            cca_result = compute_cca_at_t(data1_t, data2_t)
            cca_scores.append(cca_result[0] if not np.isnan(cca_result[0]) else 0)
            encoding_scores.append(compute_manifold_alignment_at_t(d2m, d1m))
        results[name] = {
            'rsa': rsa_scores, 'cca': cca_scores,
            'encoding': encoding_scores, 'dsa': dsa_score, 'dsa_z': dsa_z,
        }

# --- DSA point estimates (including z-score) ---
print('Computing DSA scores...')
dsa_scores   = {}   # name → 1 - true_dsa_score
dsa_zscores  = {}   # name → -z_score (higher = more similar than shuffled)
for name, (data1, data2) in comparisons.items():
    z_sc, score = compute_dsa(data1, data2)
    dsa_scores[name]  = score
    dsa_zscores[name] = z_sc
    print(f'  {name}: score={score:.4f}  z={z_sc:.2f}')
print('DSA done.')

# --- Bootstrap 95% CI on metrics (resample neurons independently) ---
# DSA z-score requires 50 permutations per call → CI for dsa_z not computed in bootstrap.
N_BOOT = 200
np.random.seed(42)
print(f'\nBootstrapping ({N_BOOT} reps, resampling neurons)...')

def compute_dsa_score(data1, data2, n_components=10):
    """DSA score only (no permutation test) for fast bootstrapping."""
    if not HAS_DSA:
        return np.nan
    N_trials = data1.shape[0]
    d1 = np.transpose(data1, (0, 2, 1)).reshape(-1, data1.shape[1])
    d2 = np.transpose(data2, (0, 2, 1)).reshape(-1, data2.shape[1])
    n_comp1 = min(n_components, d1.shape[1], N_trials * data1.shape[2] - 1)
    n_comp2 = min(n_components, d2.shape[1], N_trials * data2.shape[2] - 1)
    n_comp  = min(n_comp1, n_comp2)
    if n_comp < 1:
        return np.nan
    d1r = PCA(n_components=n_comp).fit_transform(d1).reshape(N_trials, data1.shape[2], n_comp)
    d2r = PCA(n_components=n_comp).fit_transform(d2).reshape(N_trials, data2.shape[2], n_comp)
    return 1 - np.mean(DSA(X=d1r, Y=d2r, n_delays=1).fit_score()) if HAS_DSA else np.nan

boot_ci = {}
for name, (data1, data2) in comparisons.items():
    n1 = data1.shape[1]
    n2 = data2.shape[1]
    boot_rsa, boot_cca, boot_enc, boot_dsa = [], [], [], []
    for _ in range(N_BOOT):
        idx1 = np.random.choice(n1, n1, replace=True)
        idx2 = np.random.choice(n2, n2, replace=True)
        d1b, d2b = data1[:, idx1, :], data2[:, idx2, :]
        r, c, e = compute_metrics(d1b, d2b)
        boot_rsa.append(r)
        boot_cca.append(c)
        boot_enc.append(e)
        boot_dsa.append(compute_dsa_score(d1b, d2b))
    boot_ci[name] = {
        'rsa': (np.percentile(boot_rsa, 2.5), np.percentile(boot_rsa, 97.5)),
        'cca': (np.percentile(boot_cca, 2.5), np.percentile(boot_cca, 97.5)),
        'enc': (np.percentile(boot_enc, 2.5), np.percentile(boot_enc, 97.5)),
        'dsa': (np.nanpercentile(boot_dsa, 2.5), np.nanpercentile(boot_dsa, 97.5)),
    }
    print(f'  {name} done')
print('Bootstrap complete.')

# --- Layer comparison plot with 95% CI error bars ---
layer_order  = ['L1',     'L3',     'L6',     'L8',     'L11',    'L13',    'Rec',       'Readout', 'Output']
layer_labels = ['Enc L1', 'Enc L2', 'Enc L4', 'Enc L5', 'Enc L7', 'Enc L8', 'Recurrent', 'Readout', 'Output']
metric_names = ['rsa', 'cca', 'encoding', 'dsa', 'dsa_z']

plot_mean, plot_lo, plot_hi = {}, {}, {}
for brain_area in ['V1', 'Retina']:
    plot_mean[brain_area] = {m: [] for m in metric_names}
    plot_lo[brain_area]   = {m: [] for m in metric_names}
    plot_hi[brain_area]   = {m: [] for m in metric_names}
    for layer in layer_order:
        key = f'{brain_area} vs {layer}'
        ci  = boot_ci.get(key, {})
        if key in comparisons:
            r_pt, c_pt, e_pt = compute_metrics(comparisons[key][0], comparisons[key][1])
        else:
            r_pt, c_pt, e_pt = np.nan, np.nan, np.nan
        plot_mean[brain_area]['rsa'].append(r_pt)
        plot_mean[brain_area]['cca'].append(c_pt)
        plot_mean[brain_area]['encoding'].append(e_pt)
        plot_mean[brain_area]['dsa'].append(dsa_scores.get(key, np.nan))
        plot_mean[brain_area]['dsa_z'].append(dsa_zscores.get(key, np.nan))
        for metric_key, ci_key in [('rsa','rsa'), ('cca','cca'), ('encoding','enc'), ('dsa','dsa')]:
            lo, hi = ci.get(ci_key, (np.nan, np.nan))
            plot_lo[brain_area][metric_key].append(lo)
            plot_hi[brain_area][metric_key].append(hi)
        # No bootstrap CI for dsa_z (too expensive: requires 50 permutations per rep)
        plot_lo[brain_area]['dsa_z'].append(np.nan)
        plot_hi[brain_area]['dsa_z'].append(np.nan)

x = np.arange(len(layer_order))
colors_ba = {'V1': 'steelblue', 'Retina': 'firebrick'}
markers   = {'V1': 'o',         'Retina': 's'}

avg_label = 'time-avg data' if TIME_AVERAGE else 'avg over time steps'
metric_titles = [
    f'RSA (Spearman r) [{avg_label}]',
    f'CCA (Mean Canonical Corr) [{avg_label}]',
    f'Encoding (Model → Brain R²) [{avg_label}]',
    'DSA (1 − true similarity)',
    'DSA Z-score (similarity vs shuffled baseline)',
]
fig, axes_arr = plt.subplots(5, 1, figsize=(10, 26), sharex=True)
fig.suptitle(
    f'Neural Representation Metrics vs. Model Layer\n'
    f'(95% CI via Neuron Bootstrap | TIME_AVERAGE={TIME_AVERAGE})',
    fontsize=13,
)

for i, (metric, title) in enumerate(zip(metric_names, metric_titles)):
    ax = axes_arr[i]
    for brain_area in ['V1', 'Retina']:
        y    = np.array(plot_mean[brain_area][metric], dtype=float)
        y_lo = np.array(plot_lo[brain_area][metric],   dtype=float)
        y_hi = np.array(plot_hi[brain_area][metric],   dtype=float)
        col = colors_ba[brain_area]
        ax.plot(x, y, lw=2, marker=markers[brain_area], color=col, label=brain_area, zorder=3)
        if not np.all(np.isnan(y_lo)):
            ax.fill_between(x, y_lo, y_hi, color=col, alpha=0.2, label=f'{brain_area} 95% CI')
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Score', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.axhline(0, color='k', lw=0.8, linestyle='--', alpha=0.3)

axes_arr[-1].set_xticks(x)
axes_arr[-1].set_xticklabels(layer_labels, rotation=20, ha='right', fontsize=9)
axes_arr[-1].set_xlabel('Model Layer', fontsize=10)
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
suffix = 'tavg' if TIME_AVERAGE else 'tresolved'
plt.savefig(f'../fig/additional/neural_metrics_by_layer_{suffix}.png', dpi=150, bbox_inches='tight')
plt.savefig(f'../fig/additional/neural_metrics_by_layer_{suffix}.pdf', bbox_inches='tight')
print(f'Saved → ../fig/additional/neural_metrics_by_layer_{suffix}.png')
plt.show()

# --- Save results as Markdown table ---
ci_metrics = ['rsa', 'cca', 'encoding', 'dsa']   # have bootstrap CI
metric_headers = {
    'rsa':      'RSA',
    'cca':      'CCA',
    'encoding': 'Encoding',
    'dsa':      'DSA',
    'dsa_z':    'DSA Z',
}

def fmt(val, lo=None, hi=None):
    if np.isnan(val):
        return '—'
    if lo is not None and not np.isnan(lo):
        return f'{val:.3f} [{lo:.3f}, {hi:.3f}]'
    return f'{val:.3f}'

md_lines = []
md_lines.append(f'> TIME_AVERAGE = {TIME_AVERAGE}  |  N_BOOT = {N_BOOT}')

for brain_area in ['V1', 'Retina']:
    md_lines.append(f'## {brain_area}')
    # Header
    header_cols = ['Layer'] + [metric_headers[m] for m in metric_names]
    md_lines.append('| ' + ' | '.join(header_cols) + ' |')
    # Separator (left-align all)
    md_lines.append('| ' + ' | '.join(['---'] * len(header_cols)) + ' |')
    # Rows
    for j, lbl in enumerate(layer_labels):
        row = [lbl]
        for m in metric_names:
            val = plot_mean[brain_area][m][j]
            lo  = plot_lo[brain_area][m][j]
            hi  = plot_hi[brain_area][m][j]
            row.append(fmt(val, lo if m in ci_metrics else None,
                               hi if m in ci_metrics else None))
        md_lines.append('| ' + ' | '.join(row) + ' |')
    md_lines.append('')

md_text = ''.join(md_lines)
md_path = f'../fig/additional/neural_metrics_by_layer_{suffix}.md'
with open(md_path, 'w') as fh:
    fh.write(md_text)
print(f'Saved → {md_path}')
print(md_text)


## 4. OSI and DSI: Pink Noise vs Flow Stimuli

Computes orientation selectivity index (OSI) and direction selectivity index (DSI) for all FNN neurons under directional pink noise and optical-flow stimuli, then plots histograms and scatter comparisons.

**Requires:** `fnn` package  
Source: `additional/osi_dsi.py` → `compute_and_plot_all()`  
Output: `fig/additional/osi_dsi_comparison.pdf`

In [ ]:
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
from fnn import microns
from src.plot_utils import createFlowDataset


def compute_osi(responses, angles):
    numerator   = np.abs(np.sum(responses * np.exp(1j * 2 * angles)))
    denominator = np.sum(responses)
    return numerator / (denominator + 1e-8)


def compute_dsi(responses, angles):
    numerator   = np.abs(np.sum(responses * np.exp(1j * angles)))
    denominator = np.sum(responses)
    return numerator / (denominator + 1e-8)


def save_pink_noise_video(frames, output_path='pink_noise_input.mp4', fps=30):
    T, H, W = frames.shape
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (W, H), isColor=False)
    for frame in frames:
        out.write(frame)
    out.release()
    print(f'Saved pink noise video to: {output_path}')


def generate_directional_pink_noise_frames(num_directions=16, frames_per_direction=37,
                                           frame_size=(144, 256), seed=42):
    np.random.seed(seed)
    H, W = frame_size
    angles = np.linspace(0, 2 * np.pi, num_directions, endpoint=False)

    def pink_noise(shape):
        white = np.random.randn(*shape)
        f = np.fft.fft2(white)
        fshift = np.fft.fftshift(f)
        Y, X = np.ogrid[:shape[0], :shape[1]]
        center = (shape[0] / 2, shape[1] / 2)
        dist = np.sqrt((Y - center[0])**2 + (X - center[1])**2)
        dist[dist == 0] = 1
        fshift = fshift / dist
        pink = np.fft.ifft2(np.fft.ifftshift(fshift)).real
        pink = (pink - pink.min()) / (pink.max() - pink.min()) * 255
        return pink.astype(np.uint8)

    frames = []
    direction_labels = []
    for i, angle in enumerate(angles):
        base = pink_noise(frame_size)
        flow_x = np.cos(angle)
        flow_y = np.sin(angle)
        for t in range(frames_per_direction):
            dx = int(flow_x * t * 1.5)
            dy = int(flow_y * t * 1.5)
            M = np.float32([[1, 0, dx], [0, 1, dy]])
            shifted = cv2.warpAffine(base, M, (W, H), borderMode=cv2.BORDER_REFLECT)
            frames.append(shifted)
            direction_labels.append(i)

    #save_pink_noise_video(np.array(frames, dtype=np.uint8))
    return np.array(frames, dtype=np.uint8), angles, np.array(direction_labels)


def compute_osi_dsi_per_neuron(output, angles, dir_labels):
    num_frames, num_neurons = output.shape
    num_dirs = len(angles)
    osi_list, dsi_list = [], []
    for neuron_idx in range(num_neurons):
        responses_by_dir = {k: [] for k in range(num_dirs)}
        for t in range(num_frames):
            val = output[t, neuron_idx].item() if hasattr(output[t, neuron_idx], 'item') else float(output[t, neuron_idx])
            responses_by_dir[dir_labels[t]].append(val)
        mean_responses = np.array([np.mean(responses_by_dir[i]) for i in range(num_dirs)])
        osi_list.append(compute_osi(mean_responses, angles))
        dsi_list.append(compute_dsi(mean_responses, angles))
    return np.array(osi_list), np.array(dsi_list)


def load_flow_stimuli():
    scl_factor  = 0.7
    N_INSTANCES = 3
    trial_len   = 37
    stride      = 1
    mydirs      = list(map(str, range(0, 360, 45)))
    categories  = [
        'grat_W12', 'grat_W1', 'grat_W2',
        'neg1dotflow_D1_bg', 'neg3dotflow_D1_bg', 'neg1dotflow_D2_bg', 'neg3dotflow_D2_bg',
        'pos1dotflow_D1_bg', 'pos3dotflow_D1_bg', 'pos1dotflow_D2_bg', 'pos3dotflow_D2_bg',
    ]
    flow_dataset = createFlowDataset(categories, '../stimuli/flowstims', mydirs,
                                     (800, 600), (144, 256), scl_factor, N_INSTANCES,
                                     trial_len, stride)[0]
    return np.array([frame.reshape(144, 256) for frame in flow_dataset], dtype='uint8')


def compute_flow_osi_dsi_max_per_neuron(model, angles, device):
    print('Loading flow stimuli...')
    flow_frames = load_flow_stimuli()
    T_total = flow_frames.shape[0]
    num_categories, num_directions = 11, 8
    frames_per_trial = T_total // (num_categories * num_directions)
    assert T_total % (num_categories * num_directions) == 0, 'Unexpected number of frames'
    all_osi, all_dsi = [], []
    for cat_idx in range(num_categories):
        start_idx = cat_idx * num_directions * frames_per_trial
        end_idx   = (cat_idx + 1) * num_directions * frames_per_trial
        cat_frames = flow_frames[start_idx:end_idx]
        dir_labels = np.array([d for d in range(num_directions) for _ in range(frames_per_trial)])
        with torch.no_grad():
            output = model.predict(stimuli=cat_frames)
        osi, dsi = compute_osi_dsi_per_neuron(output, angles, dir_labels)
        all_osi.append(osi)
        all_dsi.append(dsi)
    all_osi = np.stack(all_osi)
    all_dsi = np.stack(all_dsi)
    return np.mean(all_osi, axis=0), np.mean(all_dsi, axis=0)


def plot_comparison_hist_and_scatter(pink_osi, pink_dsi, flow_osi, flow_dsi,
                                     save_path='../fig/additional/osi_dsi_comparison.pdf'):
    from tueplots import bundles, axes as tpaxes
    mean_osi      = np.mean(pink_osi)
    mean_dsi      = np.mean(pink_dsi)
    mean_osi_flow = np.mean(flow_osi)
    mean_dsi_flow = np.mean(flow_dsi)
    with plt.rc_context({**bundles.neurips2024(), **tpaxes.lines()}):
        plt.figure(figsize=(16, 10))
        plt.subplot(2, 3, 1)
        plt.hist(pink_osi, bins=30, color='steelblue', edgecolor='k')
        plt.title(f'Pink Noise OSI\nMean = {mean_osi:.3f}', fontsize=20)
        plt.axvline(mean_osi, color='red', linestyle='--')
        plt.xlabel('OSI', fontsize=20); plt.ylabel('Neurons', fontsize=20)
        plt.xticks(fontsize=15); plt.yticks(fontsize=15)

        plt.subplot(2, 3, 2)
        plt.hist(pink_dsi, bins=30, color='darkorange', edgecolor='k')
        plt.title(f'Pink Noise DSI\nMean = {mean_dsi:.3f}', fontsize=20)
        plt.axvline(mean_dsi, color='red', linestyle='--')
        plt.xlabel('DSI', fontsize=20); plt.ylabel('Neurons', fontsize=20)
        plt.xticks(fontsize=15); plt.yticks(fontsize=15)

        plt.subplot(2, 3, 4)
        plt.hist(flow_osi, bins=30, color='purple', edgecolor='k')
        plt.title(f'Flow Stimuli OSI\nMean = {mean_osi_flow:.3f}', fontsize=20)
        plt.axvline(mean_osi_flow, color='red', linestyle='--')
        plt.xlabel('OSI', fontsize=20); plt.ylabel('Neurons', fontsize=20)
        plt.xticks(fontsize=15); plt.yticks(fontsize=15)

        plt.subplot(2, 3, 5)
        plt.hist(flow_dsi, bins=30, color='forestgreen', edgecolor='k')
        plt.title(f'Flow Stimuli DSI\nMean = {mean_dsi_flow:.3f}', fontsize=20)
        plt.axvline(mean_dsi_flow, color='red', linestyle='--')
        plt.xlabel('DSI', fontsize=20); plt.ylabel('Neurons', fontsize=20)
        plt.xticks(fontsize=15); plt.yticks(fontsize=15)

        plt.subplot(2, 3, 3)
        plt.scatter(pink_osi, flow_osi, alpha=0.5, color='steelblue')
        plt.plot([0, 1], [0, 1], 'r--')
        plt.xlabel('Pink OSI', fontsize=20); plt.ylabel('Flow OSI', fontsize=20)
        plt.title('OSI Comparison', fontsize=20)
        plt.xticks(fontsize=15); plt.yticks(fontsize=15)

        plt.subplot(2, 3, 6)
        plt.scatter(pink_dsi, flow_dsi, alpha=0.5, color='steelblue')
        plt.plot([0, 1], [0, 1], 'r--')
        plt.xlabel('Pink DSI', fontsize=20); plt.ylabel('Flow DSI', fontsize=20)
        plt.title('DSI Comparison', fontsize=20)
        plt.xticks(fontsize=15); plt.yticks(fontsize=15)

        plt.tight_layout()
        plt.savefig(save_path)
        plt.close()
        print(f'Saved \u2192 {save_path}')


# --- Run ---

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Loading model...')
model, ids = microns.scan(session=8, scan_idx=5)
model.to(device)
model.eval()

print('Generating pink noise...')
pink_frames, pink_angles, pink_dir_labels = generate_directional_pink_noise_frames()
with torch.no_grad():
    pink_output = model.predict(stimuli=pink_frames)
pink_osi, pink_dsi = compute_osi_dsi_per_neuron(pink_output, pink_angles, pink_dir_labels)

flow_angles = np.linspace(0, 2 * np.pi, 8, endpoint=False)
flow_osi, flow_dsi = compute_flow_osi_dsi_max_per_neuron(model, flow_angles, device)

plot_comparison_hist_and_scatter(pink_osi, pink_dsi, flow_osi, flow_dsi)
print('Done.')
